# Applying Periodic Boundary Conditions

In this notebook, we will see how to apply periodic boundary conditions to the 3D grid of region segmentation from the earlier tutorial. So if you haven't tried the 'Region segmentation' tutorial, please do so before proceeding with this one. In this tutorial, we will simply rush through the steps with no explanation up until the point where we have the 3D grid of region segmentation.

Periodic Boundary Conditions (PBC) is useful for improving our intuitive understanding of the cavities in a material, averaging the pore properties correctly, and makes it possible to work with one unit cell of the material instead of having to replicate the unit cell multiple times (for materials where the structure can be treated as rigid).

Applying PBC, is probably the most error prone step in the cavity type detection workflow, since materials can have a variety of pore structures and the application of PBC can be tricky as we will see shortly. 

Imports and paths

In [ ]:
import sys
import os
# * Put your path here
path_to_mgr = os.path.abspath(
    'c:/Users/Arun Gopalan/OneDrive - The University of Manchester/my_cloned_repos/mofography')
sys.path.append(path_to_mgr)

import mofography as mgr

path_to_cif = os.path.join(path_to_mgr, 'examples',
                           'TestCIFs', 'P1-IRMOF-1.cif')


## Read in the CIF file

In [ ]:
from ase.io import read 
mat_atoms = read(path_to_cif)
mat_updated = mgr.update_with_covalent_bonds(mat_atoms)

## Compute the distance grid

As before we are computing a 0.25 $\AA$ distance grid.

In [ ]:
dgrid = mgr.dgrid_from_atoms(mat_updated, spacing=0.25)

## Segment the distance grid

Again, as in the previous tutorial, we will use Watershed segmentation to segment the distance grid into regions using the wrapper function in mofography. We will use a `mask_thickness` of 1.5 $\AA$ as before.

In [ ]:
regions, maxima = mgr.regions_from_dgrid(dgrid.compute(), mask_thickness=1.5)

## The meaning of PBC in this context

To understand a bit more clearly what I mean by applying PBC, let's visualize the current segmentation using a 1.75 $\AA$ isosurface like before

In [ ]:
import plotly.graph_objects as go 
import plotly.express as px 

cscale = px.colors.qualitative.Pastel

isosurface_trace= mgr.plot_surface_dgrid(dgrid.compute(), ase_atoms=mat_updated,isovalue=1.75, cscale=cscale, opacity=1, stylize=True, region_labels=regions)
cell_trace  = mgr.plot_cell(mat_updated)

fig = go.Figure(data=[isosurface_trace, cell_trace])

fig.update_layout(mgr.dark_layout)
fig.show(renderer='notebook_connected')

As you can see there are 8 pockets in this unit cell
- One at the center
- 1/8 at each corner, and 8 corners to the unit cell: 1/8 * 8 = 1
- 1/4 at each edge, and 12 edges to the unit cell: 1/4 * 12 = 3
- 1/2 at each face, and 6 faces to the unit cell: 1/2 * 6 = 3

Intuitively, we know this. And in order to average the properties correctly, our algorithm needs to recognize such information. mofography has a collection of novel algorithms implemented in it to check whether pairs of regions are connected by periodic boundary conditions or not. To summarize, PBC asks the following question: 

*Which segmented pockets at the wall come together to form a complete cavity as per periodic boundary conditions?*

Let's simply run the PBC algorithm on the segmentation and see what happens. Note that the output is another 3D grid of the same size as the input, where each voxel is assigned a label corresponding to the periodic group it belongs to. Each group is a set of regions that are connected by PBC. One additionaly thing we need to do this estimation in mofography is the size of the local maxima, which we already have from the distance grid.

In [ ]:
maxima_radii  = dgrid.compute()[tuple(maxima.T)]
regions_pbc = mgr.apply_pbc(regions, maxima, maxima_radii, mat_atoms)

Now, let's visualize the results again using a 1.75 $\AA$ isosurface

In [ ]:
import plotly.graph_objects as go
import plotly.express as px

cscale = px.colors.qualitative.Pastel

isosurface_trace = mgr.plot_surface_dgrid(dgrid.compute(), ase_atoms=mat_updated,
                                          isovalue=1.75, cscale=cscale, opacity=1, stylize=True, region_labels=regions_pbc)
cell_trace = mgr.plot_cell(mat_updated)

fig = go.Figure(data=[isosurface_trace, cell_trace])

fig.update_layout(mgr.dark_layout)
fig.show(renderer='notebook_connected')


Now everything in the same pbc groups is colored the same. We can also look at the PBC groups, like this 

In [ ]:
pbc_groups = mgr.apply_pbc(regions, maxima, maxima_radii, mat_atoms, return_conn=True)
pbc_groups

We can see that there are 8 groups, and each group has 1-8 pockets, depending on whether it is at the corner, edge, face or center. This is exactly what we expect.

## Tunable parameters

There are basically two tunable parameters in the PBC algorithm, the `wall_thickness` and the `minimum_overlap_fraction`. The `wall_thickness` is the thickness of the wall in angstroms that will be considered when looking to for regions to connect via PBC. The `minimum_overlap_fraction` is the minimum fraction of wall projection of one region that needs to overlap with the wall projection of another region on the opposite wall for them to be considered connected by PBC. Note that this is checked in both directions, so if the overlap fraction is 0.5, then the regions need to overlap by at least 0.5 in at least one of the directions.

Feel free to experiment with these parameters, but the default values should work well for most cases.

## A bit more complex example

The IRMOF-1 structure was great in recognizing the PBC connected pockets, but the pore structure in materials can be way more complex. Let's put mofography to test to one of the more complex cases of NU-1000

In [ ]:

path_to_cif = os.path.join(path_to_mgr, 'examples','TestCIFs', 'NU-1000_S1.cif')

mat_atoms = read(path_to_cif)
mat_updated = mgr.update_with_covalent_bonds(mat_atoms)
dgrid = mgr.dgrid_from_atoms(mat_updated, spacing=0.25)
regions, maxima = mgr.regions_from_dgrid(dgrid.compute(), mask_thickness=1.5)
maxima_radii  = dgrid.compute()[tuple(maxima.T)]
regions_pbc = mgr.apply_pbc(regions, maxima, maxima_radii, mat_atoms)
frame_trace = mgr.plot_framework_as_mesh(mat_updated)
isosurface_trace = mgr.plot_surface_dgrid(dgrid.compute(), isovalue=1.75,ase_atoms=mat_updated,cscale=cscale, opacity=1, stylize=True, region_labels=regions_pbc)
cell_trace = mgr.plot_cell(mat_updated)
go.Figure(data=[isosurface_trace, cell_trace, *frame_trace], layout=mgr.white_layout).show(renderer='notebook_connected')

As you can see, there are a few PBC groups in NU-1000. There is a large mesopore running along the c axis at each corner. Two micro pores running along the c direction fully in the unit cell and 6 intersection pockets that connect the micro pores to the mesopore. As mofography detects them correctly. We leave as an exercise to you to visualize the original segmentation and to look at the PBC groups, to convince yourself that this is correct, as we did earlier with IRMOF-1

## Where to next?

From here you can move to the Detecting Pocket Types tutorial where will use our knowledge of PBC and some geometric information to arrive at features from which the pocket types can be detected via unsupervised machine learning.
